# 第304章: 解きほぐされた表現（Disentangled Representation）

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] Disentanglement（解きほぐし）の定義と重要性を説明できる
- [ ] β-VAEの損失関数を理解し、βの効果を実験的に確認できる
- [ ] 標準VAEとβ-VAEの潜在走査を比較して、解きほぐしの度合いを視覚的に評価できる
- [ ] Disentanglementの定量的指標（DCI、MIG等）の概念を説明できる
- [ ] 画像変容タスクにおけるDisentanglementの実用的価値を理解できる

## 🎯 前提知識

- ✅ Notebook 300-303（潜在空間の基礎、探索、ベクトル演算、走査）
- ✅ Notebook 37-38（VAE理論 — 特にELBOとKL項）

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★★☆（上級）
🎓 **カテゴリ**: 理論・実践

---

## 🌟 はじめに

前章（303）で、標準VAEの潜在空間は**もつれている（entangled）**ことを確認しました。
1つの次元を動かすと、複数の属性が同時に変わってしまう問題です。

### 🤔 なぜ「解きほぐし」が重要なのか？

**画像変容の文脈**で考えてみましょう：

```
  もつれた潜在空間                    解きほぐされた潜在空間
  ────────────────                    ──────────────────────
  「年齢だけ変えたい」                 「年齢だけ変えたい」
  → z₃を動かす                       → z₃を動かす
  → 年齢 + 髪型 + 表情が変わる 😢     → 年齢だけが変わる 🎉
```

Disentanglementが達成されると：
- **個別属性の精密制御**が可能になる
- **未知の組み合わせ**も生成できる（見たことがない「笑顔の老人」など）
- **解釈可能性**が向上する

### 📝 この章の構成

1. **Disentanglementの定義** — 数学的な定義と直感
2. **β-VAE** — KL項の重みで解きほぐしを促す手法
3. **実験: β値の影響** — 異なるβで学習して比較
4. **潜在走査による評価** — 視覚的な解きほぐしの確認
5. **定量的評価指標** — DCI、MIGの概念
6. **再構成品質とのトレードオフ** — 解きほぐしのコスト

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)

print(f"日本語フォント: {font_used}")
print(f"Device: {device}")
print("✅ 環境設定完了")

---

## 1. Disentanglementの定義

### 📊 直感的な定義

> 潜在空間がdisentangledであるとは、**各潜在変数 zᵢ が、データの生成因子のうち
> たった1つだけに対応している**状態を指す。

### 🔢 数学的な理解

データ x が K 個の独立した生成因子 v = (v₁, v₂, ..., v_K) から生成されるとします：

$$x = g(v_1, v_2, ..., v_K)$$

Disentangled表現 z = E(x) は、各 zᵢ が**1つの vⱼ にのみ**依存する表現です：

$$\forall i, \exists j: \quad z_i = f(v_j)$$

### MNISTの例

| 生成因子 v | 具体例 | 理想的なz |
|-----------|--------|---------|
| v₁: 数字の種類 | 0, 1, 2, ..., 9 | z₁ |
| v₂: 太さ | 細い ↔ 太い | z₂ |
| v₃: 傾き | 左傾き ↔ 右傾き | z₃ |
| v₄: 大きさ | 小さい ↔ 大きい | z₄ |

理想的には、z₁を動かすと**数字の種類だけ**が変わり、z₂を動かすと**太さだけ**が変わります。

---

## 2. β-VAE — 解きほぐしを促す手法

### 📊 標準VAEからβ-VAEへ

標準VAEの損失関数：
$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathbb{E}[-\log p(x|z)]}_{\text{再構成誤差}} + \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{KL正則化}}$$

**β-VAE**は、KL項に重み **β** を掛けるだけのシンプルな改良です：

$$\mathcal{L}_{\beta\text{-VAE}} = \mathbb{E}[-\log p(x|z)] + \beta \cdot D_{KL}(q(z|x) \| p(z))$$

| β値 | 効果 |
|-----|------|
| β = 1 | 標準VAE（ELBOそのまま） |
| β > 1 | KL項をより強く → 各次元がN(0,1)に近づく → 解きほぐしが促進 |
| β < 1 | KL項を弱く → 再構成重視 → もつれやすい |
| β >> 1 | KLが支配的 → 潜在空間が潰れる → 再構成品質が低下 |

### 🤔 なぜβ > 1で解きほぐされるか？

βを大きくすると、**各次元が独立にN(0,1)に近づく圧力**が強くなります。
このとき、限られた「KLの予算」を効率的に使うために、
各次元が**異なる1つの属性**を担当するように自然と分業が起きます。

In [ ]:
# ============================================================
# β-VAEモデルとデータ準備
# ============================================================

class BetaVAE(nn.Module):
    """β-VAE: βの値を変えられるVAE"""

    def __init__(self, latent_dim=10):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# データ
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("✅ モデルとデータ準備完了")

In [ ]:
# ============================================================
# 異なるβ値でVAEを学習
# β = 0.5, 1.0, 4.0, 10.0 の4パターンを比較
# ============================================================

def train_beta_vae(beta, n_epochs=20, latent_dim=10):
    """指定したβでVAEを学習"""
    model = BetaVAE(latent_dim=latent_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    history = {'recon_loss': [], 'kl_loss': [], 'total_loss': []}

    for epoch in range(n_epochs):
        model.train()
        epoch_recon, epoch_kl = 0, 0
        for x, _ in train_loader:
            x = x.view(-1, 784).to(device)
            recon, mu, logvar = model(x)

            recon_loss = nn.functional.binary_cross_entropy(recon, x, reduction='sum')
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            loss = recon_loss + beta * kl_loss  # ← β がここで効く！

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()

        n = len(train_dataset)
        history['recon_loss'].append(epoch_recon / n)
        history['kl_loss'].append(epoch_kl / n)
        history['total_loss'].append((epoch_recon + beta * epoch_kl) / n)

        if (epoch + 1) % 5 == 0:
            print(f"    Epoch {epoch+1:2d} | Recon: {epoch_recon/n:.2f}, KL: {epoch_kl/n:.2f}")

    return model, history

# 4つのβ値で学習
betas = [0.5, 1.0, 4.0, 10.0]
models = {}
histories = {}

for beta in betas:
    print(f"\n{'='*50}")
    print(f"  β = {beta} で学習")
    print(f"{'='*50}")
    models[beta], histories[beta] = train_beta_vae(beta, n_epochs=20)

print("\n✅ 全モデルの学習完了！")

In [ ]:
# ============================================================
# β値ごとの再構成品質を比較
# ============================================================

test_images = next(iter(test_loader))[0][:10].view(-1, 784).to(device)

fig, axes = plt.subplots(len(betas) + 1, 10, figsize=(16, (len(betas)+1) * 1.6))

# 元画像
for i in range(10):
    axes[0, i].imshow(test_images[i].cpu().reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('元画像', fontsize=11, rotation=0, labelpad=45)

# 各βでの再構成
for row, beta in enumerate(betas):
    models[beta].eval()
    with torch.no_grad():
        recon, _, _ = models[beta](test_images)
    for i in range(10):
        axes[row+1, i].imshow(recon[i].cpu().reshape(28, 28), cmap='gray')
        axes[row+1, i].axis('off')
    axes[row+1, 0].set_ylabel(f'β={beta}', fontsize=11, rotation=0, labelpad=35)

fig.suptitle('β値と再構成品質の関係\nβが大きいほど再構成がぼやける（KL正則化が強い）',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_304_01_beta_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 観察:")
print("   β=0.5: シャープ（再構成重視、もつれやすい）")
print("   β=1.0: 標準的なバランス")
print("   β=4.0: 少しぼける（解きほぐし促進）")
print("   β=10:  かなりぼける（過度なKL正則化）")

---

## 3. 潜在走査で解きほぐしを視覚的に評価

β-VAEの真価は**潜在走査**で明らかになります。
各β値のモデルで潜在走査を行い、1次元で何が変わるかを比較しましょう。

In [ ]:
# ============================================================
# β値ごとの潜在走査マップを比較
# ============================================================

def decode_z(model, z_vec):
    with torch.no_grad():
        z_t = torch.tensor(z_vec, dtype=torch.float32).unsqueeze(0).to(device)
        return model.decode(z_t).cpu().numpy().reshape(28, 28)

# テストデータのエンコード
def get_base_z(model, digit=3):
    model.eval()
    for x, y in test_loader:
        mask = y == digit
        if mask.any():
            x_d = x[mask][0:1].view(1, 784).to(device)
            with torch.no_grad():
                mu, _ = model.encode(x_d)
            return mu.cpu().numpy().flatten()

n_steps = 9
z_range = np.linspace(-3, 3, n_steps)

# 2つのβ (1.0 vs 4.0) を並列比較（最も重要な5次元）
fig, axes = plt.subplots(10, n_steps, figsize=(14, 16))

z_base_1 = get_base_z(models[1.0], digit=3)
z_base_4 = get_base_z(models[4.0], digit=3)

for dim in range(5):
    # β=1.0（上側）
    for i, val in enumerate(z_range):
        z_mod = z_base_1.copy()
        z_mod[dim] = val
        img = decode_z(models[1.0], z_mod)
        axes[dim, i].imshow(img, cmap='gray')
        axes[dim, i].axis('off')
        if i == 0:
            axes[dim, i].set_ylabel(f'β=1\nz{dim}', fontsize=9, rotation=0, labelpad=30)
        if dim == 0:
            axes[dim, i].set_title(f'{val:.1f}', fontsize=9)

    # β=4.0（下側）
    for i, val in enumerate(z_range):
        z_mod = z_base_4.copy()
        z_mod[dim] = val
        img = decode_z(models[4.0], z_mod)
        axes[dim + 5, i].imshow(img, cmap='gray')
        axes[dim + 5, i].axis('off')
        if i == 0:
            axes[dim + 5, i].set_ylabel(f'β=4\nz{dim}', fontsize=9, rotation=0, labelpad=30)

fig.suptitle('潜在走査の比較: β=1.0（上5行）vs β=4.0（下5行）\nβ=4.0のほうが各次元の変化が独立的になっているか？',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_304_02_traversal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 比較のポイント:")
print("   β=1.0: 1つの次元を動かすと複数の属性が同時に変化しやすい")
print("   β=4.0: 各次元がより独立した属性（太さだけ、傾きだけ）を制御している傾向")

In [ ]:
# ============================================================
# 次元ごとのKLダイバージェンス分析
# 各次元がどれだけ「使われている」かを測定
# ============================================================

def compute_kl_per_dim(model, data_loader):
    """各次元のKLダイバージェンスを計算"""
    model.eval()
    all_mu, all_logvar = [], []
    with torch.no_grad():
        for x, _ in data_loader:
            x = x.view(-1, 784).to(device)
            mu, logvar = model.encode(x)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())

    all_mu = torch.cat(all_mu)
    all_logvar = torch.cat(all_logvar)

    # 次元ごとのKL: E[-0.5 * (1 + logvar - mu² - exp(logvar))]
    kl_per_dim = -0.5 * (1 + all_logvar - all_mu.pow(2) - all_logvar.exp()).mean(dim=0)
    return kl_per_dim.numpy()

fig, axes = plt.subplots(1, len(betas), figsize=(18, 5))

for idx, beta in enumerate(betas):
    kl = compute_kl_per_dim(models[beta], test_loader)
    colors = ['steelblue' if k > 0.1 else 'lightgray' for k in kl]
    axes[idx].bar(range(len(kl)), kl, color=colors, alpha=0.8, edgecolor='gray')
    axes[idx].set_xlabel('潜在次元', fontsize=11)
    axes[idx].set_ylabel('KLダイバージェンス', fontsize=11)
    axes[idx].set_title(f'β = {beta}', fontsize=14, fontweight='bold')
    axes[idx].set_xticks(range(10))

    active = sum(1 for k in kl if k > 0.1)
    axes[idx].text(0.95, 0.95, f'活性: {active}/10',
                  transform=axes[idx].transAxes, ha='right', va='top', fontsize=11,
                  bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('次元ごとのKLダイバージェンス\n値が大きい次元 = 情報を保持している（活性次元）',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_304_03_kl_per_dimension.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 観察:")
print("   β=0.5: 多くの次元が活性 → 情報が分散（もつれやすい）")
print("   β=10:  少数の次元だけ活性 → 情報が集約（各次元の役割が明確）")
print("   「潰れた次元」= KLがほぼ0 → N(0,1)に潰されて何も表現していない")

---

## 4. 定量的評価指標

視覚的な評価だけでなく、Disentanglementを**数値で測定**する指標があります。

### 主な指標

| 指標 | 正式名称 | 何を測るか | 計算方法 |
|------|---------|----------|---------|
| **DCI** | Disentanglement, Completeness, Informativeness | 各zᵢが1つのvⱼだけに情報的か | 各次元のエントロピー |
| **MIG** | Mutual Information Gap | 最大と2番目の相互情報量の差 | MI(zᵢ, vⱼ)のギャップ |
| **β-VAE metric** | — | 固定因子を分類できるか | 線形分類器の精度 |
| **Factor VAE metric** | — | 因子ごとの多数決精度 | 多数決分類器 |

### 簡易実装: 「属性予測精度」による評価

完全なDCI/MIGを計算するには生成因子の真値が必要ですが、
MNISTでは**数字ラベル**を「生成因子の1つ」として使い、簡易的に評価できます。

> 各潜在次元 zᵢ だけから数字ラベルをどれだけ予測できるか？
> → 特定の次元に数字情報が集中しているほど disentangled

In [ ]:
# ============================================================
# 簡易Disentanglement評価
# 各次元から数字ラベルをどれだけ予測できるか
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def evaluate_disentanglement(model, data_loader):
    """各次元の数字予測精度でdisentanglement度を評価"""
    model.eval()
    all_z, all_y = [], []
    with torch.no_grad():
        for x, y in data_loader:
            mu, _ = model.encode(x.view(-1, 784).to(device))
            all_z.append(mu.cpu().numpy())
            all_y.append(y.numpy())

    z = np.concatenate(all_z)
    y = np.concatenate(all_y)

    # 各次元の予測精度
    dim_accuracies = []
    for d in range(z.shape[1]):
        z_d = z[:, d:d+1]
        scaler = StandardScaler()
        z_d_scaled = scaler.fit_transform(z_d)
        clf = LogisticRegression(max_iter=500, random_state=42)
        clf.fit(z_d_scaled[:5000], y[:5000])
        acc = clf.score(z_d_scaled[5000:], y[5000:])
        dim_accuracies.append(acc)

    # 全次元を使った予測精度
    scaler_all = StandardScaler()
    z_all_scaled = scaler_all.fit_transform(z)
    clf_all = LogisticRegression(max_iter=500, random_state=42)
    clf_all.fit(z_all_scaled[:5000], y[:5000])
    total_acc = clf_all.score(z_all_scaled[5000:], y[5000:])

    return dim_accuracies, total_acc

# 各βモデルで評価
results = {}
for beta in betas:
    dim_acc, total_acc = evaluate_disentanglement(models[beta], test_loader)
    results[beta] = {'dim_acc': dim_acc, 'total_acc': total_acc}
    print(f"β={beta:4.1f} | 全次元精度: {total_acc:.3f} | "
          f"最高次元精度: {max(dim_acc):.3f} (z{np.argmax(dim_acc)})")

In [ ]:
# ============================================================
# Disentanglement評価結果の可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左: 各次元の精度分布（β別）
x_pos = np.arange(10)
width = 0.2
colors_beta = {0.5: '#e74c3c', 1.0: '#3498db', 4.0: '#2ecc71', 10.0: '#9b59b6'}

for i, beta in enumerate(betas):
    offset = (i - 1.5) * width
    axes[0].bar(x_pos + offset, results[beta]['dim_acc'],
               width, label=f'β={beta}', color=colors_beta[beta], alpha=0.8)

axes[0].set_xlabel('潜在次元', fontsize=12)
axes[0].set_ylabel('数字分類精度', fontsize=12)
axes[0].set_title('各次元の数字予測精度', fontsize=14, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'z{i}' for i in range(10)])
axes[0].legend(fontsize=10)
axes[0].axhline(y=0.1, color='gray', linestyle='--', alpha=0.3, label='ランダム')

# 右: 集中度（最大精度 / 平均精度）
concentration = []
for beta in betas:
    accs = results[beta]['dim_acc']
    conc = max(accs) / (np.mean(accs) + 1e-8)
    concentration.append(conc)

axes[1].bar([f'β={b}' for b in betas], concentration,
           color=[colors_beta[b] for b in betas], alpha=0.8)
axes[1].set_ylabel('集中度 (max / mean)', fontsize=12)
axes[1].set_title('情報の集中度\n高い = 特定次元に数字情報が集約 = disentangled',
                  fontsize=13, fontweight='bold')

for i, (beta, conc) in enumerate(zip(betas, concentration)):
    axes[1].text(i, conc + 0.05, f'{conc:.2f}', ha='center', fontsize=12)

plt.tight_layout()
plt.savefig('fig_304_04_disentanglement_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 集中度が高いほど、数字の情報が少数の次元に集約されている")
print("   → 他の次元は太さ、傾きなど別の属性を担当している可能性が高い")

---

## 5. 再構成品質とのトレードオフ

β-VAEの重要な特性は、**Disentanglement と 再構成品質のトレードオフ**です。

```
  再構成品質    │*
  （高い）     │ *
               │  *
               │    *
               │       *
               │           *______
  （低い）     │─────────────────→ β
               0.5  1.0   4.0  10.0
  
  Disentangle  │          ___*____
  メント       │       *
  （高い）     │    *
               │  *
  （低い）     │*
               │─────────────────→ β
```

**最適なβ**はタスク依存です：
- 再構成が重要 → β ≈ 1
- 属性操作が重要 → β ≈ 4-10

In [ ]:
# ============================================================
# トレードオフの可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 再構成品質（MSE）
recon_errors = []
for beta in betas:
    models[beta].eval()
    mse = 0
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.view(-1, 784).to(device)
            recon, _, _ = models[beta](x)
            mse += nn.functional.mse_loss(recon, x, reduction='sum').item()
    recon_errors.append(mse / len(test_dataset))

axes[0].plot(betas, recon_errors, 'o-', color='steelblue', linewidth=2, markersize=10)
axes[0].set_xlabel('β', fontsize=14)
axes[0].set_ylabel('再構成誤差 (MSE)', fontsize=12)
axes[0].set_title('β vs 再構成品質', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# Disentanglement度（集中度）
axes[1].plot(betas, concentration, 'o-', color='#2ecc71', linewidth=2, markersize=10)
axes[1].set_xlabel('β', fontsize=14)
axes[1].set_ylabel('情報集中度', fontsize=12)
axes[1].set_title('β vs Disentanglement', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')

fig.suptitle('β-VAEのトレードオフ\n再構成品質 ↔ Disentanglement',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_304_05_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 トレードオフのまとめ:")
print(f"   β=0.5: 再構成 {recon_errors[0]:.4f}, 集中度 {concentration[0]:.2f}")
print(f"   β=1.0: 再構成 {recon_errors[1]:.4f}, 集中度 {concentration[1]:.2f}")
print(f"   β=4.0: 再構成 {recon_errors[2]:.4f}, 集中度 {concentration[2]:.2f}")
print(f"   β=10:  再構成 {recon_errors[3]:.4f}, 集中度 {concentration[3]:.2f}")

---

## まとめ

### 🎯 このノートブックで学んだこと

**Disentanglementの概念**
- ✓ 各潜在次元がデータの1つの生成因子にだけ対応する状態
- ✓ 画像変容の属性制御に不可欠

**β-VAEの仕組み**
- ✓ KL項にβ > 1の重みを掛けることで解きほぐしを促進
- ✓ 限られたKL予算の中で次元が自然と「分業」する

**トレードオフ**
- ✓ β↑ → Disentanglement↑ だが 再構成品質↓
- ✓ タスクに応じた最適なβの選択が重要

**定量的評価**
- ✓ 次元ごとの属性予測精度で集中度を測定
- ✓ DCI、MIG等のより高度な指標の概念を理解

### ⚠️ よくある間違い

❌ 「β-VAEは常に標準VAEより良い」
✅ 再構成品質は犠牲になる。画像生成の品質が重要なタスクでは標準VAEのほうが良い場合もある

❌ 「βを大きくすればするほどdisentangleされる」
✅ βが大きすぎると潜在空間が完全に潰れ、何も表現できなくなる。最適なβは有限

### ✅ 学習チェックリスト

- [ ] β-VAEとVAEの損失関数の違いを書けるか？
- [ ] βが大きいと解きほぐしが促進される理由を説明できるか？
- [ ] 再構成品質とdisentanglementのトレードオフを理解しているか？

---

**次のステップ**: ノートブック305「GAN潜在空間の構造」で、GANの潜在空間の特性とStyleGANのW空間を学びます。Phase 1（基礎）からPhase 2（GAN応用）へ進みます！

---

## 🎓 自己評価クイズ

### Q1: β-VAEの損失関数で β=1 のとき、何に等しくなるか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 標準VAEの損失関数（負のELBO）に等しくなる

β=1は特別なケースで、β-VAEは標準VAEの一般化です。
β>1でKL正則化を強化し、β<1で弱化します。

</details>

---

### Q2: KLダイバージェンスがほぼ0の次元は何を意味するか？

<details>
<summary>💡 答えを見る</summary>

**答え**: その次元が事前分布 N(0,1) とほぼ同じになっており、データの情報を保存していない

「潰れた次元」と呼ばれます。デコーダはこの次元の値を無視して画像を生成しています。
βが大きいほど、より多くの次元が潰れる傾向があります。

</details>

---

### Q3: β=4のモデルの走査で、z₃を動かすと太さだけが変わり、z₅を動かすと傾きだけが変わった。これは良いことか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 非常に良い — これが理想的なDisentanglementの状態

各次元が独立した1つの属性だけを制御しているため、属性の個別操作が可能です。
画像変容において「年齢だけ変える」「表情だけ変える」といった精密な制御ができます。

</details>